In [13]:
import polars as pl
from utils.db_connection import get_connection
import plotly.express as px
import plotly.graph_objects as go
import duckdb

In [14]:
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)

polars.config.Config

In [15]:
conn = get_connection()

### Data Parquet

In [16]:
df = pl.scan_parquet("data/raw/movimientos_*.parquet")

print("columnas:", df.collect_schema().names())

print("filas totales:", df.select(pl.len()).collect().item())

print("id_cliente únicos:", df.select(pl.col("id_cliente").n_unique()).collect().item())

columnas: ['id_cliente', 'date', 'product', 'type', 'fund', 'amount', 'description', 'commercial_name']
filas totales: 99000
id_cliente únicos: 3000


## EDA with SAMPLE: id_cliente  CLI002977

### El objetivo es ver como se ejecuta paso a paso el pipeline desde que se carga 

### 1. RAW

In [17]:
# Para efectos practicos vamos a filtrar por un cliente en particular y ver sus transacciones ordenadas por fecha
# En el esquiema RAW, se carga la data tal como llega esta tabla tiene particionada el campo: _loaded_at (EXPL) PT by M
# Y tambien indices a los campos id_cliente, fund y type.

# ¿ Porque particicionar y crear indices ? Porque si la data es por millones de registros esto nos permite
# hacer consultas mas rapidas y eficientes. TODO: Se puede usar sharding por tabla. BQ

# Metadatos de la tabla: _loaded_at, id_cliente, _source_file, _batch_id. Se agregan por la razon que esto nos permite
# Aplicar filtros y tecnicas de SCD2 y aplicar auditoria a la data.


with conn.cursor() as cur:
    cur.execute("""
        SELECT *
        FROM "raw".transactions
        WHERE id_cliente = 'CLI002977'
        ORDER BY date;
    """)

    rows = cur.fetchall()
    columns = [desc.name for desc in cur.description]

df_mv = pl.DataFrame(rows, schema=columns, orient="row")

duck = duckdb.connect()

duck.register("transactions", df_mv)

result = duck.sql("""
    SELECT COUNT(*) as total_transactions
    FROM transactions
""").show()

# result = duck.sql("""
#     SELECT *
#     FROM transactions
#     ORDER BY date ASC LIMIT 20
# """).show()

┌────────────────────┐
│ total_transactions │
│       int64        │
├────────────────────┤
│                 31 │
└────────────────────┘



### 2. STAGING

In [18]:
# -- Limpieza y normalización del raw. Decisiones de calidad de datos documentadas:
# --   * id_cliente nulo/vacío -> se descarta (no se puede rastrear sin cliente)
# --   * amount no numérico -> queda NULL tras el cast seguro (no se descarta la
# --     fila completa, para no perder trazabilidad; se puede filtrar abajo)
# --   * strings vacíos -> se normalizan a NULL
# --   * description, type, amount -> se normaliza a minúsculas para evitar duplicar categorías por casting
# --
# -- HALLAZGO CRÍTICO: el parquet fuente NO trae ningún identificador único por
# -- transacción. "id_cliente" es el identificador del CLIENTE (se repite ~16-17
# -- veces por cliente en un solo corte: 50,000 filas / 3,000 clientes únicos).
# -- Usarlo tal cual como unique_key del snapshot SCD2 colapsaba todas las
# -- transacciones de un mismo cliente en una sola fila "vigente", perdiendo
# -- datos reales silenciosamente.
# --
# -- Solución: se construye un transaction_id sintético a partir de una LLAVE
# -- DE NEGOCIO (los campos que identifican la transacción y que, según el
# -- enunciado, NO deberían cambiar entre cortes): id_cliente + date + product
# -- + fund + type. Los campos que el enunciado explícitamente señala como
# -- corregibles (amount, description, "u otro campo") quedan FUERA de la
# -- llave y son los que entran al row_hash para detectar correcciones.
# -- Supuesto documentado: se asume que esta combinación es única por cliente
# -- dentro de un mismo corte (validar con el dataset real antes de producción).


with conn.cursor() as cur2:
    cur2.execute("""
        SELECT *
        FROM "staging".stg_transactions
        WHERE id_cliente = 'CLI002977'
        ORDER BY movement_date, _loaded_at;
    """)

    rows = cur2.fetchall()
    columns = [desc.name for desc in cur2.description]

df_stg = pl.DataFrame(rows, schema=columns, orient="row")

duck = duckdb.connect()

duck.register("stg_transactions", df_stg)

result = duck.sql("""
    SELECT COUNT(*) as total_transactions
    FROM stg_transactions
""").show()

# result = duck.sql("""
#     SELECT *
#     FROM stg_transactions
#     ORDER BY movement_date, _batch_id LIMIT 20
# """).show()

┌────────────────────┐
│ total_transactions │
│       int64        │
├────────────────────┤
│                 29 │
└────────────────────┘



### 3. Snapshot

In [19]:
# -- El snapshot ve SOLO el estado del corte más reciente (el que run_pipeline.py
# -- acaba de cargar). Así dbt compara "estado de hoy" contra "estado de ayer":
# --   * transaction_id nuevo            -> se inserta su primera versión
# --   * mismo transaction_id, otro row_hash (amount/description/commercial_name)
# --                                     -> se cierra la versión anterior y se abre otra
# --   * transaction_id que estaba y ya no viene en este corte
# --                                     -> invalidate_hard_deletes cierra la fila (dbt_valid_to)
# --   * mismo row_hash                  -> no se toca
# --
# -- Si leyera todos los cortes acumulados, una transacción eliminada seguiría
# -- apareciendo en la fuente (con los datos del corte viejo) y nunca se detectaría.
# --
# -- transaction_id es sintético (ver stg_transactions.sql): el parquet no trae
# -- un id único por transacción, solo id_cliente (identificador del cliente).
# -- stg_transactions ya deduplica por llave de negocio dentro de cada corte.

with conn.cursor() as cur3:
    cur3.execute("""
        SELECT *
        FROM "snapshots".snapshot_transactions
        WHERE id_cliente = 'CLI002977'
        ORDER BY movement_date, _loaded_at;
    """)

    rows = cur3.fetchall()
    columns = [desc.name for desc in cur3.description]

df_snp = pl.DataFrame(rows, schema=columns, orient="row")

duck = duckdb.connect()

duck.register("snapshot", df_snp)

result = duck.sql("""
    SELECT COUNT(*) as total_transactions
    FROM snapshot
""").show()

# result = duck.sql("""
#     SELECT *
#     FROM stg_transactions
#     ORDER BY movement_date, _batch_id LIMIT 10
# """).show()

┌────────────────────┐
│ total_transactions │
│       int64        │
├────────────────────┤
│                 21 │
└────────────────────┘



### 4. Marts

In [20]:
# -- Histórico completo (todas las versiones de todos los ids) con una
# -- clasificación explícita del tipo de evento, para auditoría y para las
# -- consultas de "insights" (cuántos registros nuevos/corregidos/eliminados
# -- hubo entre cada par de cortes).

# -- Histórico completo (todas las versiones de todos los ids) con una
# -- clasificación explícita del tipo de evento, para auditoría y para las
# -- consultas de "insights" (cuántos registros nuevos/corregidos/eliminados
# -- hubo entre cada par de cortes).
# --
# -- Nota: dbt solo agrega la columna dbt_is_deleted al snapshot a partir de
# -- la SEGUNDA corrida de `dbt snapshot` (la primera vez no hay nada contra
# -- qué comparar para detectar eliminaciones). Este modelo lo detecta en
# -- tiempo de compilación para no romperse si solo se ha procesado un corte.


with conn.cursor() as cur4:
    cur4.execute("""
        SELECT *
        FROM "marts".fct_historical_transactions
        WHERE id_cliente = 'CLI002977'
        ORDER BY movement_date;
    """)

    rows = cur4.fetchall()
    columns = [desc.name for desc in cur4.description]

df_mrt = pl.DataFrame(rows, schema=columns, orient="row")

duck = duckdb.connect()

duck.register("fct_historical_transactions", df_mrt)

result = duck.sql("""
    SELECT COUNT(*) as total_transactions
    FROM fct_historical_transactions
""").show()

# result = duck.sql("""
#     SELECT *
#     FROM fct_historical_transactions
#     ORDER BY movement_date, _batch_id
# """).show()

┌────────────────────┐
│ total_transactions │
│       int64        │
├────────────────────┤
│                 21 │
└────────────────────┘



In [21]:
# Current transactions

with conn.cursor() as cur9:
    cur9.execute("""
        SELECT *
        FROM "marts".dim_current_transactions
        ORDER BY movement_date;
    """)

    rows = cur9.fetchall()
    columns = [desc.name for desc in cur9.description]

df_curr = pl.DataFrame(rows, schema=columns, orient="row")

duck = duckdb.connect()

duck.register("dim_current_transactions", df_curr)

result = duck.sql("""
    SELECT COUNT(*) as total_transactions
    FROM dim_current_transactions
""").show()

┌────────────────────┐
│ total_transactions │
│       int64        │
├────────────────────┤
│              48895 │
└────────────────────┘



## Insights

In [22]:
# ¿Como se ve el negocio hoy?
#   Este insight muestra el volumen de movimientos y monto total, agrupado por producto,
#   fondo y tipo, sobre el estado ACTUAL (no histórico)

with conn.cursor() as cur5:
    cur5.execute("""
        SELECT * FROM "marts".insights_by_product ;
    """)

    rows = cur5.fetchall()
    columns = [desc.name for desc in cur5.description]

df_ins = pl.DataFrame(rows, schema=columns, orient="row")

duck = duckdb.connect()

duck.register("insights_by_product", df_ins)

result = duck.sql("""
    SELECT sum(transaction_count) FROM insights_by_product;
""").show()

# result = duck.sql("""
#     SELECT *
#     FROM insights_by_product ORDER BY transaction_count DESC LIMIT 10
# """).show()

┌────────────────────────┐
│ sum(transaction_count) │
│         int128         │
├────────────────────────┤
│                  48895 │
└────────────────────────┘



In [23]:
# ¿Como evolucionaron los datos de un dia a otro?
# cuántos registros nuevos, corregidos y eliminados hubo en cada corte procesado

with conn.cursor() as cur6:
    cur6.execute("""
        SELECT * FROM "marts".insights_by_change_trx ;
    """)

    rows = cur6.fetchall()
    columns = [desc.name for desc in cur6.description]

df_trx = pl.DataFrame(rows, schema=columns, orient="row")

duck = duckdb.connect()

duck.register("insights_by_change_trx", df_trx)

result = duck.sql("""
    SELECT *
    FROM insights_by_change_trx
""").show()

┌──────────────────────────────────────┬────────────────────────────┬────────────┬─────────────┐
│              _batch_id               │        _source_file        │ event_type │ event_count │
│               varchar                │          varchar           │  varchar   │    int64    │
├──────────────────────────────────────┼────────────────────────────┼────────────┼─────────────┤
│ b406bbfb-bbfc-4be6-b220-4de70a1440a8 │ movimientos_dia_T1.parquet │ corrected  │       13861 │
│ 4c348c24-93c4-4719-9142-776272427bc8 │ movimientos_dia_T.parquet  │ corrected  │       46884 │
│ 4c348c24-93c4-4719-9142-776272427bc8 │ movimientos_dia_T.parquet  │ new        │        3000 │
└──────────────────────────────────────┴────────────────────────────┴────────────┴─────────────┘



In [24]:
# Volumen de transacciones por producto y fondo
with conn.cursor() as cur7:
    cur7.execute("""
        SELECT
            product,
            fund,
            SUM(transaction_count) AS transaction_count
        FROM marts.insights_by_product
        GROUP BY product, fund
        ORDER BY product, fund;
    """)

    rows = cur7.fetchall()
    columns = [desc.name for desc in cur7.description]

# PostgreSQL → Polars
df_trxs = pl.DataFrame(
    rows,
    schema=columns,
    orient="row"
)

# DuckDB
duck = duckdb.connect()

duck.register(
    "insights_by_trxs",
    df_trxs
)

result = duck.sql("""
    SELECT *
    FROM insights_by_trxs
    ORDER BY product, fund
""")

result.show()

# --------------------------------------------------
# Pivot: producto x fondo
# --------------------------------------------------

pivot = (
    df_trxs
    .pivot(
        on="fund",
        index="product",
        values="transaction_count",
        aggregate_function="sum",
    )
    .fill_null(0)
    .sort("product")
)

# --------------------------------------------------
# Heatmap
# --------------------------------------------------

fig = go.Figure(
    data=go.Heatmap(
        z=pivot.drop("product").to_numpy(),
        x=pivot.drop("product").columns,
        y=pivot["product"].to_list(),
        colorscale="Blues",
        hovertemplate=(
            "Producto: %{y}<br>"
            "Fondo: %{x}<br>"
            "Transacciones: %{z:,.0f}"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    title="Volumen de transacciones por producto x fondo",
    height=460,
    xaxis_title="Fondo",
    yaxis_title="Producto",
)

fig.show()

┌────────────────────┬───────────────────┬───────────────────┐
│      product       │       fund        │ transaction_count │
│      varchar       │      varchar      │   decimal(38,0)   │
├────────────────────┼───────────────────┼───────────────────┤
│ Acciones           │ balanceado        │               792 │
│ Acciones           │ conservador       │               862 │
│ Acciones           │ crecimiento       │               885 │
│ Acciones           │ internacional     │               868 │
│ Acciones           │ mercado monetario │               841 │
│ Acciones           │ renta fija        │               917 │
│ Acciones           │ renta variable    │               903 │
│ Bonos              │ balanceado        │               872 │
│ Bonos              │ conservador       │               863 │
│ Bonos              │ crecimiento       │               828 │
│   ·                │      ·            │                ·  │
│   ·                │      ·            │             

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
# ¿hacia dónde se mueve el dinero, y cuándo?
# entradas positivas, salidas negativas, para ver tendencia neta en el tiempo

with conn.cursor() as cur8:
    cur8.execute("""
        SELECT
            movement_date,
            product,
            SUM(
                CASE
                    WHEN type = 'entrada' THEN amount
                    ELSE 0
                END
            ) AS total_entradas,
            SUM(
                CASE
                    WHEN type = 'salida' THEN amount
                    ELSE 0
                END
            ) AS total_salidas,
            SUM(
                CASE
                    WHEN type = 'entrada' THEN amount
                    WHEN type = 'salida' THEN -amount
                    ELSE 0
                END
            ) AS net_flow
        FROM "marts".dim_current_transactions
        WHERE movement_date IS NOT NULL
        GROUP BY movement_date, product
        ORDER BY movement_date, product
    """)

    rows = cur8.fetchall()
    columns = [desc.name for desc in cur8.description]

# PostgreSQL → Polars
df_net = pl.DataFrame(rows, schema=columns, orient="row")

# DuckDB
duck = duckdb.connect()
duck.register("insights_fin_net", df_net)

result = duck.sql("""
    SELECT *
    FROM insights_fin_net
    LIMIT 10
""")

result.show()

# --------------------------------------------------
# Gráfico
# --------------------------------------------------

fig = go.Figure()

# Obtener productos
products = df_net["product"].unique().sort()

for product in products:

    group = (
        df_net.filter(pl.col("product") == product)
        # .head(10)
        .sort("movement_date")
    )

    fig.add_trace(
        go.Scatter(
            x=group["movement_date"].to_list(),
            y=group["net_flow"].to_list(),
            mode="lines+markers",
            name=str(product),
            hovertemplate=(
                "Fecha: %{x}<br>"
                "Flujo neto: %{y:,.0f}"
                "<extra>%{fullData.name}</extra>"
            ),
        )
    )

# Línea de referencia en 0
fig.add_hline(y=0, line_dash="dot", line_color="gray")

fig.update_layout(
    title="Flujo neto de caja por producto (entradas - salidas)",
    xaxis_title="Fecha",
    yaxis_title="Flujo neto",
    height=460,
    legend_title="Producto",
    hovermode="x unified",
)

fig.show()

┌───────────────┬────────────────────┬────────────────┬───────────────┬───────────────┐
│ movement_date │      product       │ total_entradas │ total_salidas │   net_flow    │
│     date      │      varchar       │ decimal(38,2)  │ decimal(38,2) │ decimal(38,2) │
├───────────────┼────────────────────┼────────────────┼───────────────┼───────────────┤
│ 2024-09-15    │ Acciones           │  1763125103.97 │ 1968671030.28 │ -205545926.31 │
│ 2024-09-15    │ Bonos              │  1857674854.31 │ 1583751159.72 │  273923694.59 │
│ 2024-09-15    │ CDT                │  1697264485.36 │ 1541993548.03 │  155270937.33 │
│ 2024-09-15    │ Cuenta de Ahorro   │  1946870590.69 │ 1722895399.69 │  223975191.00 │
│ 2024-09-15    │ Divisas            │  1797143004.36 │ 1568379258.70 │  228763745.66 │
│ 2024-09-15    │ ETF                │  2029318564.60 │ 1572860634.48 │  456457930.12 │
│ 2024-09-15    │ Fondo de Inversión │  2129140287.94 │ 1933597801.52 │  195542486.42 │
│ 2024-09-15    │ Fondo de Pensi